# Pitch Compensation, Mooring Reposition — Same Kind of Control?

Reproduces all three Part 4 demos directly from the real control interfaces: the wind-filtered, Ct-table-feedforward-plus-rate-feedback pitch controller (a genuine in-run feedback loop, toggled via configure_pitch_controller), the collective blade pitch (CBC) gain schedule and command trace behind it, and the mooring-reposition solver (a between-run proposal). Neither controller commands real hardware -- run_simulation only ever simulates the consequence of a proposed configuration.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import plotly.graph_objects as go
from controller import (
    DEFAULT_FL_KP_SCHEDULE,
    ControlConfig,
    RunStore,
    configure_pitch_controller,
    run_simulation,
    solve_equilibrium_lengths,
)
from helpers import (
    OC4_MOORING,
    CatenaryMooring,
    anchor_position,
    fairlead_world_position,
    resolve_line_force,
)
# same steady-state window as the article's figures/demo_results.json
from make_blog_figures import _DURATION, _STARTUP_S, _metrics


## Regulating Pitch

In [2]:
pitch_store = RunStore()
pitch_config = ControlConfig()
region_env = {'Hs': 3.0, 'Tp': 9.0, 'TI': 0.10}
regions = [8.0, 15.0, 20.0]
modes = ['off', 'ff', 'pi']

runs = {}
metrics = {}
for U_hub in regions:
    for mode in modes:
        configure_pitch_controller(pitch_store, pitch_config, mode=mode)
        env = {**region_env, 'U_hub': U_hub}
        out = run_simulation(
            pitch_store, pitch_config, duration=_DURATION, **env)
        runs[(U_hub, mode)] = out
        metrics[(U_hub, mode)] = _metrics(pitch_store.get(out['run_id'])['result'])

for (U_hub, mode), m in metrics.items():
    print(U_hub, mode, m['pitch_rms_ss_deg'], m['surge_rms_ss_m'])

8.0 off 0.7297678987767384 5.228366330465332
8.0 ff 0.7297678987767384 5.228366330465332
8.0 pi 0.6939177335628546 5.054754223981116
15.0 off 0.8195598237958797 5.729142850812819
15.0 ff 0.8195591395731291 5.7291206686896565
15.0 pi 0.7763870027147359 5.5423002309645595
20.0 off 0.68995425062324 4.655913796191383
20.0 ff 0.6899517306024712 4.655868179022626
20.0 pi 0.598539974537749 4.507842343745822


In [3]:
fig = go.Figure()
for U_hub in regions:
    for mode in modes:
        run_id = runs[(U_hub, mode)]['run_id']
        result = pitch_store.get(run_id)['result']
        fig.add_trace(go.Scatter(
            x=result['time'], y=np.degrees(result['pitch']),
            mode='lines', name=f'U_hub={U_hub:g} {mode}'))
fig.update_layout(title='Platform pitch: off vs ff vs pi across regions',
                   xaxis_title='time [s]', yaxis_title='pitch [deg]')
fig.show()

![Pitch control, off vs ff vs pi: platform response and control input, across three wind-speed regions](figures/fig_p4_pitch_regions.png)

## Collective Blade Pitch (CBC)

In [4]:
print('DEFAULT_FL_KP_SCHEDULE (wind_speed, fl_kp) -- the only schedule now:')
for wind_speed, fl_kp in DEFAULT_FL_KP_SCHEDULE:
    print(f'  {wind_speed} m/s: fl_kp={fl_kp}')


DEFAULT_FL_KP_SCHEDULE (wind_speed, fl_kp) -- the only schedule now:
  11.4 m/s: fl_kp=40.0
  25.0 m/s: fl_kp=40.0


In [5]:
# same real pi run reused from the region comparison above -- no re-simulation
cbc_run_id = runs[(15.0, 'pi')]['run_id']
cbc_result = pitch_store.get(cbc_run_id)['result']

fig = go.Figure()
fig.add_trace(go.Scatter(x=cbc_result['time'], y=np.degrees(cbc_result['pitch_command']),
                          mode='lines', name='phi_ref (collective)'))
fig.update_layout(title='Collective blade pitch command, pi controller, U_hub=15 m/s',
                   xaxis_title='time [s]', yaxis_title='commanded blade pitch [deg]')
fig.show()

![Three blades pitching collectively, driven by the real pi-controller command trace](figures/anim_cbc_pitch_w_plot.gif)

## Repositioning

In [6]:
# static check only -- no FMU run, same catenary force model
# controller.solve_equilibrium_lengths solves against
from dataclasses import replace

target_position = (4.0, 4.0)
nominal_lengths = [OC4_MOORING.unstretched_length] * 3
platform_state = {'surge': target_position[0], 'sway': target_position[1], 'heave': 0.0,
                   'roll': 0.0, 'pitch': 0.0, 'yaw': 0.0}

fx_total, fy_total = 0.0, 0.0
for i in range(3):
    model = CatenaryMooring(cfg=replace(OC4_MOORING, unstretched_length=nominal_lengths[i]))
    anchor = anchor_position(i)
    fairlead = fairlead_world_position(platform_state, i)
    fx, fy, _fz, _t = resolve_line_force(anchor, fairlead, model)
    fx_total += fx
    fy_total += fy
    print(f'line {i+1} (nominal): fx={fx/1e3:.1f} kN, fy={fy/1e3:.1f} kN')
print(f'net at nominal lengths: fx={fx_total/1e3:.1f} kN, fy={fy_total/1e3:.1f} kN')

line 1 (nominal): fx=774.7 kN, fy=-3.9 kN
line 2 (nominal): fx=-659.5 kN, fy=-1137.5 kN
line 3 (nominal): fx=-447.0 kN, fy=762.0 kN
net at nominal lengths: fx=-331.8 kN, fy=-379.5 kN


In [7]:
# solve for lengths that zero the net force at (4, 4), changing each
# line as little as possible from nominal
proposal = solve_equilibrium_lengths(
    (0.0, 0.0), target_position=target_position,
    nominal_lengths=tuple(nominal_lengths), minimize_length_change=True)
print(proposal)

{'converged': True, 'lengths': [831.3997762615685, 841.0733077647393, 834.0011958785036], 'predicted_tensions_n': [1158950.341181931, 1147542.275002468, 1155959.7059422354]}


![Moving to +4/+4 m under nominal lengths, then adjusting lengths to zero the force](figures/anim_mooring_repositioning.gif)